In [2]:
import pandas as pd
import time
import os
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [3]:
# CSV load karo
df = pd.read_csv("INTERVIEW.csv")
print(f"✅ Dataset loaded! Total questions: {len(df)}")
print("Categories:", df['category'].unique())

# AI Model load karo (pehli baar 1-2 min lagenge)
print("\n⏳ Loading AI model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ AI Model ready!")

✅ Dataset loaded! Total questions: 15
Categories: <ArrowStringArray>
['HR', 'Technical']
Length: 2, dtype: str

⏳ Loading AI model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5179.52it/s]


✅ AI Model ready!


In [5]:
# ============================================
#        🎤 AI INTERVIEW SIMULATOR
# ============================================

def analyze_answer(user_answer, ideal_answer):
    if len(user_answer.strip()) < 3:
        return 0.0
    emb1 = model.encode([user_answer])
    emb2 = model.encode([ideal_answer])
    score = cosine_similarity(emb1, emb2)[0][0]
    return round(float(score) * 100, 2)

def get_confidence_score(answer):
    fillers = ["um", "uh", "hmm", "like", "basically",
               "actually", "literally", "you know", "kind of"]
    words = answer.lower().split()
    count = sum(words.count(w) for w in fillers)
    confidence = max(0, 100 - (count * 10))
    return confidence, count

def get_wpm(word_count, time_seconds):
    if time_seconds < 1:
        return 0
    return round(word_count / (time_seconds / 60))

def speed_feedback(wpm):
    if wpm == 0: return "⚠️ Could not measure"
    elif wpm < 80: return "🐢 Too slow"
    elif wpm <= 150: return "✅ Perfect pace!"
    elif wpm <= 180: return "🚀 Slightly fast"
    else: return "⚡ Too fast"

def overall_feedback(score):
    if score >= 85: return "🌟 Excellent! You are interview-ready!"
    elif score >= 70: return "👍 Good! A little more practice needed."
    elif score >= 50: return "📚 Average - work on your answers."
    else: return "💪 Keep practicing!"

# ============================================
# 👇 SIRF YE 2 LINES CHANGE KARO
CATEGORY = "All"       # "All" ya "HR" ya "Technical"
MY_ANSWER = "Python is a high level programming language used for web development and data science"
# ============================================

# Category filter
if CATEGORY == "HR":
    pool = df[df['category'] == 'HR']
elif CATEGORY == "Technical":
    pool = df[df['category'] == 'Technical']
else:
    pool = df

question_row = pool.sample(1).iloc[0]

print("=" * 55)
print("        🎤 AI INTERVIEW SIMULATOR")
print("=" * 55)
print(f"\n📂 Category  : {question_row['category']}")
print(f"\n❓ QUESTION  :\n\n   {question_row['question']}\n")
print(f"\n📝 Your Answer : {MY_ANSWER}\n")

# Analysis
start_time = time.time()
time_taken = 15.0  # default 15 seconds
word_count = len(MY_ANSWER.split())

answer_score = analyze_answer(MY_ANSWER, question_row['ideal_answer'])
confidence, fillers = get_confidence_score(MY_ANSWER)
wpm = get_wpm(word_count, time_taken)
s_feedback = speed_feedback(wpm)

final_score = round(
    (answer_score  * 0.50) +
    (confidence    * 0.30) +
    (min(wpm, 150) / 150 * 100 * 0.20),
    2
)

# Results
print("=" * 55)
print("           📊 YOUR RESULTS")
print("=" * 55)
print(f"  📝 Words typed      : {word_count} words")
print(f"  🚀 Speaking speed   : {wpm} WPM  → {s_feedback}")
print(f"  🎯 Answer quality   : {answer_score}%")
print(f"  😎 Confidence score : {confidence}% (Fillers: {fillers})")
print("=" * 55)
print(f"  🏆 FINAL SCORE      : {final_score}%")
print("=" * 55)
print(f"\n  {overall_feedback(final_score)}")
print("\n" + "=" * 55)
print("  💡 IDEAL ANSWER:")
print("=" * 55)
print(f"  {question_row['ideal_answer']}")
print("=" * 55)

last_result = {
    "Question"       : question_row['question'],
    "Category"       : question_row['category'],
    "User Answer"    : MY_ANSWER,
    "Words"          : word_count,
    "Time (sec)"     : time_taken,
    "WPM"            : wpm,
    "Answer Score %" : answer_score,
    "Confidence %"   : confidence,
    "Filler Words"   : fillers,
    "Final Score %"  : final_score,
    "Feedback"       : overall_feedback(final_score)
}
print("\n✅ Run Cell 4 to save report!")

        🎤 AI INTERVIEW SIMULATOR

📂 Category  : Technical

❓ QUESTION  :

   What is a dictionary in Python?


📝 Your Answer : Python is a high level programming language used for web development and data science

           📊 YOUR RESULTS
  📝 Words typed      : 14 words
  🚀 Speaking speed   : 56 WPM  → 🐢 Too slow
  🎯 Answer quality   : 25.08%
  😎 Confidence score : 100% (Fillers: 0)
  🏆 FINAL SCORE      : 50.01%

  📚 Average - work on your answers.

  💡 IDEAL ANSWER:
  A dictionary stores key value pairs and allows fast data retrieval.

✅ Run Cell 4 to save report!


In [6]:
# ============================================
#         💾 SAVE REPORT TO CSV
# ============================================

os.makedirs("reports", exist_ok=True)
report_path = "reports/interview_report.csv"

new_entry = pd.DataFrame([last_result])

# Purani report hai toh usmein add karo
if os.path.exists(report_path):
    old_data = pd.read_csv(report_path)
    final_report = pd.concat([old_data, new_entry], ignore_index=True)
else:
    final_report = new_entry

final_report.to_csv(report_path, index=False)

print("✅ Report saved successfully!")
print(f"📁 Location      : reports/interview_report.csv")
print(f"📊 Total sessions: {len(final_report)}")
print(f"📈 Average score : {round(final_report['Final Score %'].mean(), 2)}%")
print(f"🏆 Best score    : {final_report['Final Score %'].max()}%")
print("\n--- Latest Entry ---")
display(final_report.tail(3))

✅ Report saved successfully!
📁 Location      : reports/interview_report.csv
📊 Total sessions: 1
📈 Average score : 50.01%
🏆 Best score    : 50.01%

--- Latest Entry ---


,Question,Category,User Answer,Words,Time (sec),WPM,Answer Score %,Confidence %,Filler Words,Final Score %,Feedback
0,What is a dictionary in Python?,Technical,Python is a high level programming language us...,14,15.0,56,25.08,100,0,50.01,📚 Average - work on your answers.
